# 15 · MRP / supply plan (Step 7 — second planning engine, time-phased)

**Time-phased weekly MRP** over a 13-week horizon. It **consumes the inventory-optimization policy** from nb 14
(order-up-to level `S*` + safety stock) and applies it **week by week** across the horizon, then explodes the
resulting planned production through the bill of materials into dependent net requirements and planned orders.

**Consistency with nb 14 (by construction):** the finished-goods plan walks projected inventory forward and, each
week, applies nb 14's order-up-to policy (order up to `S*` when the projected balance drops below `safety_stock`),
lead-time-offset. **Week 1 of this plan is exactly nb 14's "order now?" decision**; the sum across 13 weeks is the
horizon requirement — so the two engines agree, and procurement / production / distribution get the **13-week
schedule** (when each order fires) they need, not a single 2-week trigger.

**Key modeling point:** the order-up-to level stays **lead-time-sized** (we never target 13 weeks of stock — that
would be gross overstock). The horizon is covered by **re-applying** the short policy across weekly buckets.

### Method
1. **Weekly gross requirements** — finished goods (FERT): the weekly demand forecast; components: accumulated from
   parents' planned-order *releases* (low-level coded, parents before components).
2. **Projected-inventory walk** — per week `avail = prior_projected + scheduled − gross`; if `avail < safety`,
   plan a receipt. **Make-to-stock finished** (nb 14 policy) → order up to `S*`; **make-to-order finished**
   (beverages, no stock/policy) → lot-for-lot to demand; **components** → lot-for-lot to safety.
3. **Lead-time offset** — a receipt needed in week `t` is *released* in week `t − L` (`L` = supplier or production
   lead, weeks); releases before week 1 are clamped to week 1 and flagged `past_due`.
4. **Explode** — a parent's weekly planned *release* drives its components' gross in that week (`× qty_per × (1+scrap)`).
5. **Order type** — `PURCHASE` (ROH) / `PRODUCTION` (HALB/FERT).

**Reads:** `v_material_bom_edges` (planning BOM, nb 08), `demand_twin_metrics_timeseries` (weekly FERT demand),
`inventory_twin_current` (on-hand), `procurement_twin_current` (open PO + supplier lead), `production_twin_current`
(open production + production lead), **`inventory_policy` (order-up-to `S*` + safety stock — the nb 14 policy)**,
`planning_config`. **Writes:** `monitoring.mrp_plan` (weekly grain, deterministic overwrite) + `recommended_action`
on open `INVENTORY` events.

### Scope / caveats (v1)
- **Infinite capacity** (finite scheduling is next, blocked on a routing master).
- **Open orders folded into week-1 on-hand** (no dated scheduled receipts in source) — a simplification.
- **Mixed-demand items** (green beans, sold *and* roasted): nb 14 plans the independent portion; MRP nets the
  dependent portion lot-for-lot to nb 14's safety. The two still share physical stock — reconcile in S&OP.
- **UOM** normalized within a family (g/kg, ml/l, ea); cross-family flagged `uom_mismatch`.

## Config

In [ ]:
dbutils.widgets.text("target_catalog", "mickmickstableawscoff_catalog", "Target catalog")
dbutils.widgets.text("monitoring_schema", "monitoring", "Monitoring schema")
dbutils.widgets.text("horizon_weeks", "13", "Planning horizon (weeks)")

import math, uuid
import numpy as np
import pandas as pd
from pyspark.sql import functions as F

FQ_MON = f"{dbutils.widgets.get('target_catalog').strip()}.{dbutils.widgets.get('monitoring_schema').strip()}"
H = int(dbutils.widgets.get("horizon_weeks"))
REFRESH_ID = str(uuid.uuid4())
print("MRP target:", f"{FQ_MON}.mrp_plan", "| horizon:", H, "weeks")

## Policy parameters (`planning_config` via `cfg()`)

In [ ]:
def cfg(key, default, kind="double"):
    col = "value_string" if kind == "string" else "value_double"
    try:
        r = spark.sql(f"SELECT {col} FROM {FQ_MON}.v_resolved_config WHERE param_key = '{key}'").collect()
        return r[0][0] if r and r[0][0] is not None else default
    except Exception:
        return default

MOQ = float(cfg("min_order_qty", 0.0))
MULT = float(cfg("order_multiple", 1.0))
print(f"MOQ={MOQ}  order_multiple={MULT}")

## Assemble inputs
BOM, **weekly** finished-goods demand, the supply state, and the **nb 14 policy** (`order_up_to_level`,
`safety_stock`). Small data → pandas.

In [ ]:
def num(x, d=0.0):
    if x is None: return d
    try: v = float(x)
    except (TypeError, ValueError): return d
    return d if (isinstance(v, float) and math.isnan(v)) else v

# --- BOM edges (planning-grade view, nb 08 — not gold) ---
bom = spark.table(f"{FQ_MON}.v_material_bom_edges").select(
    "parent", "parent_type", "parent_text", "plant", "comp", "comp_type",
    "comp_text", "comp_group", "qty_per", "scrap", "comp_uom").toPandas()

# --- weekly finished-goods demand: sum across channels per (material, plant, week) ---
fcw = (spark.table(f"{FQ_MON}.demand_twin_metrics_timeseries")
       .where("period_type = 'FORECAST'")
       .groupBy("material", "plant", "period_start")
       .agg(F.sum("forecast_qty").alias("qty"), F.first("uom").alias("uom")).toPandas())
fcw["week"] = pd.to_datetime(fcw["period_start"])
weeks = [pd.Timestamp(w) for w in sorted(fcw["week"].unique())[:H]]   # the H weekly buckets
widx = {w: i + 1 for i, w in enumerate(weeks)}                        # week -> 1..H
demand = {}                                                           # (m,p) -> [qty_w1..qty_wH]
for (m, p), grp in fcw.groupby(["material", "plant"]):
    arr = [0.0] * H
    for r in grp.itertuples():
        w = pd.Timestamp(r.week)
        if w in widx:                                                # ignore weeks beyond the horizon
            arr[widx[w] - 1] += num(r.qty)
    demand[(m, p)] = arr

# --- supply state ---
inv = spark.table(f"{FQ_MON}.inventory_twin_current").select(
    "material", "plant", "on_hand_qty", "uom", "material_text", "material_group").toPandas()
on_hand = {(r.material, r.plant): num(r.on_hand_qty) for r in inv.itertuples()}
stock_uom = {(r.material, r.plant): r.uom for r in inv.itertuples()}

proc = spark.table(f"{FQ_MON}.procurement_twin_current").toPandas()
_ps = proc.sort_values("po_qty", ascending=False).groupby(["material", "plant"], as_index=False).first()
proc_lead = {(r.material, r.plant): num(r.avg_lead_time_days, None) for r in _ps.itertuples()}
proc_sup = {(r.material, r.plant): (r.supplier_name or r.supplier) for r in _ps.itertuples()}
open_po = {k: num(v) for k, v in proc.groupby(["material", "plant"])["open_po_qty"].sum().items()}

prod = spark.table(f"{FQ_MON}.production_twin_current").toPandas()
prod_lead = {(r.material, r.plant): num(r.avg_actual_lead_time_hours, None) for r in prod.itertuples()}
open_prod = {(r.material, r.plant): num(r.open_production_qty) for r in prod.itertuples()}

# --- nb 14 inventory-optimization policy ---
pol = spark.table(f"{FQ_MON}.inventory_policy").select(
    "material", "plant", "order_up_to_level", "safety_stock", "replenishment_mode").toPandas()
S_star = {(r.material, r.plant): num(r.order_up_to_level, None) for r in pol.itertuples()}
safety = {(r.material, r.plant): num(r.safety_stock) for r in pol.itertuples()}

print(f"BOM edges={len(bom)}  weeks={len(weeks)}  FERT-demand series={len(demand)}  policy rows={len(pol)}")

## Time-phased explosion (weekly, low-level coded)

In [ ]:
UNIT = {"G": ("mass", 1.0), "KG": ("mass", 1000.0), "MG": ("mass", 0.001), "T": ("mass", 1e6),
        "ML": ("vol", 1.0), "L": ("vol", 1000.0), "CL": ("vol", 10.0),
        "EA": ("cnt", 1.0), "PC": ("cnt", 1.0), "ST": ("cnt", 1.0), "PAK": ("cnt", 1.0), "ROL": ("cnt", 1.0)}
def convert(qty, frm, to):
    if frm == to: return qty, False
    a, b = UNIT.get((frm or "").upper()), UNIT.get((to or "").upper())
    if a and b and a[0] == b[0]:
        return qty * a[1] / b[1], False
    return qty, True

def round_mult(q, mult):
    return round(q / mult) * mult if mult and mult > 0 else q

# nodes + graph
nodes = {}
edges = []
for r in bom.itertuples():
    pk, ck = (r.parent, r.plant), (r.comp, r.plant)
    nodes.setdefault(pk, {"type": r.parent_type, "text": r.parent_text, "group": None, "uom": None})
    n = nodes.setdefault(ck, {"type": r.comp_type, "text": r.comp_text, "group": r.comp_group, "uom": r.comp_uom})
    if n["uom"] is None: n["uom"] = r.comp_uom
    edges.append((pk, ck, num(r.qty_per), num(r.scrap)))
for k, n in nodes.items():
    if n["uom"] is None: n["uom"] = stock_uom.get(k) or "EA"

# topological level (FERT tops = 0)
level = {k: 0 for k in nodes}
for _ in range(len(nodes) + 1):
    changed = False
    for pk, ck, *_ in edges:
        if level[ck] < level[pk] + 1:
            level[ck] = level[pk] + 1; changed = True
    if not changed: break
order = sorted(nodes, key=lambda k: level[k])

# weekly gross per node
gross = {k: ([demand.get(k, [0.0] * H)[t] for t in range(H)] if nodes[k]["type"] == "FERT"
             else [0.0] * H) for k in nodes}

rows = []
for k in order:
    m, p = k; n = nodes[k]; puom = n["uom"]; mtype = n["type"]
    # starting position: on-hand + open orders (folded to week 1) converted to planning uom
    oo = open_prod.get(k, 0.0) if mtype != "ROH" else open_po.get(k, 0.0)
    start_raw = on_hand.get(k, 0.0) + oo
    start, mm = convert(start_raw, stock_uom.get(k, puom), puom)
    ss, _ = convert(safety.get(k, 0.0), stock_uom.get(k, puom), puom)
    sstar_raw = S_star.get(k)
    sstar = None
    if sstar_raw is not None:
        sstar, _ = convert(sstar_raw, stock_uom.get(k, puom), puom)
    use_ots = (mtype == "FERT") and (sstar is not None)   # make-to-stock finished with nb14 order-up-to

    if mtype == "ROH":
        lt = proc_lead.get(k); otype = "PURCHASE"
    else:
        lt = prod_lead.get(k); otype = "PRODUCTION"
        if lt is not None: lt = lt / 24.0
    Lw = max(1, math.ceil((lt if lt is not None else 7.0) / 7.0))

    g = gross[k]
    proj = [0.0] * H; receipt = [0.0] * H; release = [0.0] * H
    prev = start
    for t in range(H):
        avail = prev - g[t]
        if use_ots:
            rcv = (sstar - avail) if avail < ss else 0.0
        else:                                   # lot-for-lot to safety (components / make-to-order)
            rcv = (ss - avail) if avail < ss else 0.0
        rcv = max(0.0, rcv)
        if rcv > 0:
            rcv = max(round_mult(rcv, MULT), MOQ)
        proj[t] = avail + rcv; receipt[t] = rcv; prev = proj[t]
        if rcv > 0:
            rel_t = max(0, t - Lw)              # release Lw weeks earlier (clamp to week 1)
            release[rel_t] += rcv

    past_due = release[0] > 0 and any(receipt[t] > 0 and (t - Lw) < 0 for t in range(H))

    # explode this parent's weekly releases into children's gross (component needed at release week)
    for pk, ck, qp, scr in edges:
        if pk == k:
            for t in range(H):
                if release[t] > 0:
                    gross[ck][t] += release[t] * qp * (1.0 + scr)

    for t in range(H):
        rows.append(dict(material=m, plant=p, material_text=n["text"], material_group=n["group"],
                         material_type=mtype, bom_level=level[k], planning_uom=puom,
                         week_index=t + 1, week_start=weeks[t].date() if hasattr(weeks[t], "date") else weeks[t],
                         gross_requirement=g[t], projected_on_hand=proj[t],
                         planned_order_receipt=receipt[t], planned_order_release=release[t],
                         order_type=otype, lead_time_weeks=Lw,
                         demand_source=("INDEPENDENT" if mtype == "FERT" else "DEPENDENT"),
                         order_up_to_level=sstar, safety_stock=ss, uom_mismatch=bool(mm),
                         past_due=bool(past_due),
                         primary_supplier=(proc_sup.get(k) if mtype == "ROH" else None)))

mrp = pd.DataFrame(rows)
print("mrp rows (material x week):", len(mrp))
_tot = mrp.groupby("material_type")["planned_order_release"].sum()
print(_tot.to_string())

## Write `monitoring.mrp_plan` (weekly grain, deterministic overwrite)

In [ ]:
from pyspark.sql.types import (StructType, StructField, StringType, DoubleType,
                                   IntegerType, BooleanType, DateType)
SCHEMA = StructType([
    StructField("material", StringType()), StructField("plant", StringType()),
    StructField("material_text", StringType()), StructField("material_group", StringType()),
    StructField("material_type", StringType()), StructField("bom_level", IntegerType()),
    StructField("planning_uom", StringType()), StructField("week_index", IntegerType()),
    StructField("week_start", DateType()), StructField("gross_requirement", DoubleType()),
    StructField("projected_on_hand", DoubleType()), StructField("planned_order_receipt", DoubleType()),
    StructField("planned_order_release", DoubleType()), StructField("order_type", StringType()),
    StructField("lead_time_weeks", IntegerType()), StructField("demand_source", StringType()),
    StructField("order_up_to_level", DoubleType()), StructField("safety_stock", DoubleType()),
    StructField("uom_mismatch", BooleanType()), StructField("past_due", BooleanType()),
    StructField("primary_supplier", StringType()),
])
ORDER = [f.name for f in SCHEMA.fields]
INT_COLS = {"bom_level", "week_index", "lead_time_weeks"}
BOOL_COLS = {"uom_mismatch", "past_due"}
STR_COLS = {"material", "plant", "material_text", "material_group", "material_type", "planning_uom",
            "order_type", "demand_source", "primary_supplier"}
def _norm(c, v):
    if c == "week_start": return v
    if v is None or (isinstance(v, float) and math.isnan(v)): return None
    if c in INT_COLS: return int(v)
    if c in BOOL_COLS: return bool(v)
    if c in STR_COLS: return str(v)
    return float(v)
recs = [tuple(_norm(c, row.get(c)) for c in ORDER) for row in mrp.to_dict("records")]
out = (spark.createDataFrame(recs, schema=SCHEMA)
       .withColumn("refresh_id", F.lit(REFRESH_ID)).withColumn("generated_at", F.current_timestamp()))
(out.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{FQ_MON}.mrp_plan"))
spark.sql(f"ALTER TABLE {FQ_MON}.mrp_plan SET TBLPROPERTIES ('comment' = "
          "'Time-phased weekly MRP over the planning horizon: applies nb 14 order-up-to policy per week, explodes "
          "the BOM into dependent planned orders (PURCHASE/PRODUCTION). Week 1 reconciles with the inventory-policy "
          "reorder decision. v1 infinite-capacity. See implementation plan + inventory_policy_model.html.')")
print("wrote", spark.table(f"{FQ_MON}.mrp_plan").count(), "rows to", f"{FQ_MON}.mrp_plan")

## Write recommendations back onto open inventory events
Summarize the 13-week schedule per item onto its open `INVENTORY` event.

In [ ]:
spark.sql(f"""
MERGE INTO {FQ_MON}.events e
USING (
  SELECT material, plant, MAX(order_type) AS order_type,
         SUM(planned_order_release) AS total_release,
         MIN(CASE WHEN planned_order_release > 0 THEN week_index END) AS first_week,
         SUM(CASE WHEN planned_order_release > 0 THEN 1 ELSE 0 END) AS n_orders,
         MAX(planning_uom) AS uom
  FROM {FQ_MON}.mrp_plan GROUP BY material, plant
) r
  ON e.function = 'INVENTORY' AND e.status = 'open'
     AND e.entity_id = r.material AND e.plant = r.plant
WHEN MATCHED THEN UPDATE SET e.recommended_action = concat(coalesce(e.recommended_action, ''),
  ' | ', CASE WHEN r.total_release > 0
    THEN concat('MRP 13w: ', CAST(r.n_orders AS STRING), ' planned ', r.order_type, ' order(s) totalling ',
                CAST(round(r.total_release, 0) AS STRING), ' ', r.uom, ', first in week ', CAST(r.first_week AS STRING), '.')
    ELSE 'MRP 13w: no orders (projected stock covers demand over the horizon).' END)
""")
print("MRP 13-week schedule merged onto inventory events.")

## Verify

In [ ]:
print("=== week-1 (should reconcile with nb 14 inventory_policy suggested order) ===")
display(spark.sql(f"""
SELECT m.material_text, m.material_type, ROUND(m.planned_order_receipt,0) mrp_wk1_receipt,
       ROUND(p.suggested_order_qty,0) nb14_suggested, p.replenishment_mode
FROM {FQ_MON}.mrp_plan m
LEFT JOIN {FQ_MON}.inventory_policy p USING (material, plant)
WHERE m.week_index = 1 AND m.material_type = 'FERT' ORDER BY m.material_text"""))
print("=== 13-week planned-order schedule (finished goods + green beans) ===")
display(spark.sql(f"""
SELECT material_text, order_type, week_index, ROUND(planned_order_release,0) release
FROM {FQ_MON}.mrp_plan
WHERE planned_order_release > 0 AND (material_type='FERT' OR material_group='BEANS_GREEN')
ORDER BY material_text, week_index"""))
print("=== horizon totals per item ===")
display(spark.sql(f"""
SELECT material_text, material_type, order_type, ROUND(SUM(planned_order_release),0) total_planned,
       MIN(CASE WHEN planned_order_release>0 THEN week_index END) first_wk, MAX(uom_mismatch) uom_mismatch
FROM {FQ_MON}.mrp_plan GROUP BY material_text, material_type, order_type
ORDER BY material_type, material_text"""))

## Next
The plan is now **time-phased and consistent with nb 14** (week 1 = the inventory-policy decision; the horizon =
the full 13-week schedule). Next: **finite production scheduling** (sequence the `PRODUCTION` releases against
work-centre capacity — blocked on a routing master), **distribution planning**, then **S&OP/IBP** reconciliation
(incl. the mixed-demand shared-stock fix) committing a plan-of-record (Step 8, Lakebase). `# TODO — EXTEND`:
dated scheduled receipts, capacity checks, lot-sizing policies, pegging.